# Lista 1

**Aluno:** [Eduardo Maciel Alexandre](mailto:ema2@ic.ufal.br)
\
**Nome da base:** "Telco-Customer-Churn.csv"

## Orientações

- Escolha apenas uma das bases disponíveis e resolva todas as 10 questões usando essa mesma base.
- Desenvolva toda a atividade em Python, no formato de entrega do Google Colab.
- Organize o notebook por questão, com códigos executáveis, saídas geradas e comentários objetivos.
- Não troque de base ao longo da atividade.
- Não faça tratamento manual linha por linha.
- Sempre que necessário, sustente suas decisões com tabelas, métricas, gráficos e resultados do código.

## Importação das bibliotecas

In [2]:
# Bibliotecas principais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Pré-processamento e modelagem
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)

# Configuração visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## Carregamento da base

In [3]:
# Carregamento da base escolhida para toda a atividade
df = pd.read_csv('./data/Telco-Customer-Churn.csv')

# Visualização inicial
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Questão 1 – Diagnóstico estrutural

**Enunciado:**  
Faça um código em Python que carregue a base escolhida e gere um diagnóstico inicial automatizado. Mostre dimensões da base, tipos de dados, valores ausentes, duplicidades, cardinalidade das colunas e possíveis inconsistências de leitura. Em seguida, defina qual será o problema computacional tratado no cenário escolhido, deixando claro se a tarefa será de classificação, regressão ou segmentação. Organize essa etapa em uma função reutilizável.

### Raciocínio
A estratégia foi criar uma função única de diagnóstico para evitar análises manuais coluna a coluna. Essa função calcula automaticamente: dimensões, tipos de dados, ausências, duplicidades e cardinalidade. Além disso, inclui uma auditoria inicial de inconsistências de leitura, procurando padrões comuns em bases reais como textos com espaços extras, valores vazios em colunas textuais e colunas numéricas lidas como texto (caso clássico de `TotalCharges` no Telco).

### Desenvolvimento
Implementar uma função reutilizável `structural_diagnosis`, aplicá-la ao `df` e apresentar as tabelas-resumo + definição formal do problema computacional.

In [ ]:
# Código da Questão 1
def structural_diagnosis(df: pd.DataFrame, target: str) -> dict:
    summary = {}

    # 1) Dimensões
    summary['dimensions'] = {'linhas': df.shape[0], 'colunas': df.shape[1]}

    # 2) Tipos de dados
    data_types = df.dtypes.astype(str).rename('dtype').to_frame()

    # 3) Valores ausentes
    missing_abs = df.isna().sum().rename('ausentes_abs') # Contagem absoluta de ausentes
    missing_pct = (df.isna().mean() * 100).round(2).rename('ausentes_pct') # Percentual de ausentes

    # 4) Cardinalidade
    cardinality = df.nunique(dropna=False).rename('cardinalidade')

    # 5) Quadro estrutural consolidado
    structural_table = pd.concat([data_types, missing_abs, missing_pct, cardinality], axis=1) # Concatenação das informações estruturais
    structural_table = structural_table.sort_values(by=['ausentes_abs', 'cardinalidade'], ascending=False)
    summary['structural_table'] = structural_table

    # 6) Duplicidades
    summary['duplicates_abs'] = int(df.duplicated().sum()) # Contagem absoluta de duplicatas
    summary['duplicates_pct'] = round(float(df.duplicated().mean() * 100), 2) # Percentual de duplicatas

    # 7) Inconsistências de leitura
    inconsistencies = []
    for column in df.columns:
        series = df[column]

        if pd.api.types.is_string_dtype(series):
            text_series = series.astype(str)

            # Espaços em excesso (início/fim)
            trim_diff = (text_series.fillna('') != text_series.fillna('').str.strip()).sum()
            if trim_diff > 0:
                inconsistencies.append({
                    'coluna': column,
                    'tipo_problema': 'espacos_extras',
                    'qtd_registros': int(trim_diff),
                    'detalhe': 'Valores com espaços no início/fim.'
                })

            # Vazios textuais ("", " ", etc.)
            text_empty = text_series.str.strip().eq('').sum()
            if text_empty > 0:
                inconsistencies.append({
                    'coluna': column,
                    'tipo_problema': 'vazio_textual',
                    'qtd_registros': int(text_empty),
                    'detalhe': 'Campos vazios mascarados como string.'
                })

            # Possível numérica lida como texto
            numeric_try = pd.to_numeric(text_series.str.strip(), errors='coerce')
            non_null = series.notna().sum()
            convertible = numeric_try.notna().sum()

            if non_null > 0 and (convertible / non_null) >= 0.8:
                inconsistencies.append({
                    'coluna': column,
                    'tipo_problema': 'numerica_em_texto',
                    'qtd_registros': int(convertible),
                    'detalhe': 'Coluna majoritariamente numérica armazenada como texto.'
                })

    inconsistencies_df = pd.DataFrame(inconsistencies)
    if not inconsistencies_df.empty:
        inconsistencies_df = inconsistencies_df.sort_values(by=['qtd_registros', 'coluna'], ascending=False)

    summary['inconsistencies'] = inconsistencies_df

    # 8) Definição do problema computacional
    target_classes = sorted(str(c) for c in df[target].dropna().unique())
    summary['problem'] = {
        'tipo': 'Classificação binária',
        'variavel_alvo': target,
        'classes_observadas': sorted(target_classes),
        'objetivo': 'Prever se um cliente irá cancelar (Churn = Yes) ou permanecer (Churn = No).'
    }

    return summary


q1_results = structural_diagnosis(df, target='Churn')

print('Dimensões da base:', q1_results['dimensions'])
print(f"Duplicatas: {q1_results['duplicates_abs']} ({q1_results['duplicates_pct']}%)")
print('\nDefinição do problema computacional:')
for key, value in q1_results['problem'].items():
    print(f'- {key}: {value}')

print('\nTop 10 colunas no diagnóstico estrutural:')
display(q1_results['structural_table'].head(10))

print('Possíveis inconsistências de leitura:')
if q1_results['inconsistencies'].empty:
    print('Nenhuma inconsistência detectada pelas regras automáticas.')
else:
    display(q1_results['inconsistencies'])

Dimensões da base: {'linhas': 7043, 'colunas': 21}
Duplicatas: 0 (0.0%)

Definição do problema computacional:
- tipo: Classificação binária
- variavel_alvo: Churn
- classes_observadas: ['No', 'Yes']
- objetivo: Prever se um cliente irá cancelar (Churn = Yes) ou permanecer (Churn = No).

Top 10 colunas no diagnóstico estrutural:


,dtype,ausentes_abs,ausentes_pct,cardinalidade
customerID,str,0,0.0,7043
TotalCharges,str,0,0.0,6531
MonthlyCharges,float64,0,0.0,1585
tenure,int64,0,0.0,73
PaymentMethod,str,0,0.0,4
MultipleLines,str,0,0.0,3
InternetService,str,0,0.0,3
OnlineSecurity,str,0,0.0,3
OnlineBackup,str,0,0.0,3
DeviceProtection,str,0,0.0,3


Possíveis inconsistências de leitura:


,coluna,tipo_problema,qtd_registros,detalhe
2,TotalCharges,numerica_em_texto,7032,Coluna majoritariamente numérica armazenada co...
0,TotalCharges,espacos_extras,11,Valores com espaços no início/fim.
1,TotalCharges,vazio_textual,11,Campos vazios mascarados como string.


### Conclusão da Questão 1
O diagnóstico estrutural confirmou que a base possui **7043 linhas e 21 colunas**, sem registros duplicados, e que o problema central será de **classificação binária** com alvo `Churn` (`Yes`/`No`). A inspeção automática também identificou uma inconsistência crítica na coluna `TotalCharges`: ela está armazenada como texto, com **11 registros vazios/espaços**, o que exige tratamento de tipagem e ausências nas próximas etapas. Assim, a base está adequada ao objetivo didático, mas depende de limpeza reproduzível antes da modelagem.

## Questão 2 – Auditoria de qualidade

**Enunciado:**  
Faça um código em Python para auditar a qualidade dos dados. Identifique valores implausíveis, colunas com tipos inadequados, categorias raras, padrões suspeitos de preenchimento, possíveis outliers e inconsistências importantes para o cenário escolhido. Se houver datas, verifique coerência temporal. Se houver valores monetários, verifique compatibilidade entre grandezas. Ao final, apresente um resumo dos principais problemas encontrados.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [5]:
# Código da Questão 2

### Conclusão da Questão 2
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 3 – Limpeza dos dados

**Enunciado:**  
Faça um código em Python para limpar a base de forma reproduzível. Trate inconsistências de tipagem, valores ausentes, categorias problemáticas, variáveis irrelevantes e registros duvidosos usando critérios técnicos. Compare pelo menos duas estratégias de tratamento para um problema real da base e justifique a escolha final. Mostre também o impacto quantitativo de cada etapa da limpeza.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [6]:
# Código da Questão 3

### Conclusão da Questão 3
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 4 – Análise exploratória

**Enunciado:**  
Faça um código em Python para desenvolver uma análise exploratória orientada ao problema central da base escolhida. Gere tabelas e visualizações que revelem padrões relevantes entre a variável principal e os demais atributos. Se o cenário envolver classificação, compare os grupos da variável-alvo. Se envolver regressão, investigue relação entre o alvo e as variáveis explicativas. Se envolver segmentação, explore possíveis estruturas de agrupamento. Apresente uma leitura analítica dos resultados obtidos.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [7]:
# Código da Questão 4

### Conclusão da Questão 4
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 5 – Associação entre variáveis

**Enunciado:**  
Faça um código em Python para medir a associação entre as variáveis explicativas e a variável principal da análise. Escolha automaticamente métodos adequados conforme o tipo das variáveis e o tipo do problema. Ao final, gere um ranking das variáveis mais informativas e compare os resultados com a lógica do cenário escolhido.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [8]:
# Código da Questão 5

### Conclusão da Questão 5
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 6 – Engenharia de atributos

**Enunciado:**  
Faça um código em Python para criar pelo menos cinco novos atributos a partir da base original. Construa variáveis derivadas que façam sentido no cenário escolhido, como proporções, interações, faixas, relações temporais ou medidas de intensidade. Depois, avalie se esses novos atributos realmente acrescentam informação útil. Organize essa etapa em uma função reutilizável.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [9]:
# Código da Questão 6

### Conclusão da Questão 6
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 7 – Pré-processamento

**Enunciado:**  
Faça um código em Python para montar um pipeline completo de pré-processamento com `Pipeline` e `ColumnTransformer`. Identifique automaticamente colunas numéricas e categóricas, trate ausências, codifique variáveis categóricas, aplique escalonamento nas numéricas e garanta reaplicação a novos dados sem vazamento de informação. Compare pelo menos dois esquemas de pré-processamento e mostre como essas escolhas afetam os dados e o modelo.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [10]:
# Código da Questão 7

### Conclusão da Questão 7
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 8 – Modelagem com KNN

**Enunciado:**  
Faça um código em Python para dividir a base em treino, validação e teste de forma adequada ao problema escolhido. Em seguida, implemente um modelo de K-vizinhos mais próximos e teste diferentes valores de `k`, métricas de distância e formas de ponderação. Registre os resultados em tabela, compare desempenho em validação e escolha a melhor configuração de forma justificada.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [11]:
# Código da Questão 8

### Conclusão da Questão 8
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 9 – Comparação de modelos

**Enunciado:**  
Faça um código em Python para avaliar o KNN em diferentes cenários de preparação dos dados, como uso ou não de padronização, presença ou ausência de atributos derivados, uso de todas as variáveis ou de um subconjunto selecionado, e tratamento do desbalanceamento quando fizer sentido. Depois, implemente um segundo modelo supervisionado e compare com o melhor KNN em métricas, tempo de execução e perfil dos erros. Conclua se o KNN é ou não uma boa escolha para a base utilizada.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [12]:
# Código da Questão 9

### Conclusão da Questão 9
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Questão 10 – Consolidação da solução

**Enunciado:**  
Faça um código em Python para consolidar toda a solução em um fluxo reutilizável. Implemente uma função que receba um novo caso em formato de dicionário ou `DataFrame` de uma linha, aplique o pipeline construído e retorne a previsão final de forma interpretável. Depois, gere um relatório resumido com a base escolhida, os principais problemas encontrados, os atributos mais relevantes, a melhor configuração do modelo, as métricas finais e uma recomendação executiva. Finalize com uma avaliação crítica sobre a maturidade da solução para uso prático.

### Raciocínio
Escreva aqui a estratégia adotada para responder à questão, explicando de forma objetiva a lógica da análise.

### Desenvolvimento
Implemente abaixo o código da questão.

In [13]:
# Código da Questão 10

### Conclusão da Questão 10
Registre aqui as principais interpretações e conclusões obtidas com base nos resultados.

## Conclusão Final

Apresente uma síntese geral do trabalho, destacando:

- principais problemas encontrados na base;
- principais decisões metodológicas;
- melhor configuração de modelo obtida;
- limitações da análise;
- avaliação final da adequação da solução ao cenário escolhido.

**Bom trabalho!**